In [12]:
from sys import base_prefix

import pandas as pd

#1.患者基本信息表(主表）
patients=pd.DataFrame({
    '患者ID':[1,2,3,4],
    '姓名':['张三','李四','王五','赵六'],
    '年龄':[45,67,52,38]
})

#2.就诊记录表（子表）
visits=pd.DataFrame({
    '患者ID':[1,2,2,3,5],
    '就诊日期':['2025-01-01','2025-01-05','2025-02-10','2025-01-20','2025-03-01'],
    '诊断':['肺炎','高血压','肺炎','糖尿病','骨折']
})

print("===患者表===")
print(patients)

print("\n===就诊表===")
print(visits)

#3.尝试4种合并方式，观察区别
print("\n=== 内连接 (Inner) - 只取交集 ===")
print(pd.merge(patients,visits,on='患者ID',how='inner'))

print("\n=== 左连接 (Left) - 保留所有患者 ===")
print(pd.merge(patients,visits,on="患者ID",how="left"))

print("\n=== 右连接 (Right) - 保留所有就诊记录 ===")
print(pd.merge(patients,visits,on='患者ID',how="right"))

print("\n=== 外连接 (Outer) - 全部保留 ===")
print(pd.merge(patients,visits,on='患者ID',how="outer"))

===患者表===
   患者ID  姓名  年龄
0     1  张三  45
1     2  李四  67
2     3  王五  52
3     4  赵六  38

===就诊表===
   患者ID        就诊日期   诊断
0     1  2025-01-01   肺炎
1     2  2025-01-05  高血压
2     2  2025-02-10   肺炎
3     3  2025-01-20  糖尿病
4     5  2025-03-01   骨折

=== 内连接 (Inner) - 只取交集 ===
   患者ID  姓名  年龄        就诊日期   诊断
0     1  张三  45  2025-01-01   肺炎
1     2  李四  67  2025-01-05  高血压
2     2  李四  67  2025-02-10   肺炎
3     3  王五  52  2025-01-20  糖尿病

=== 左连接 (Left) - 保留所有患者 ===
   患者ID  姓名  年龄        就诊日期   诊断
0     1  张三  45  2025-01-01   肺炎
1     2  李四  67  2025-01-05  高血压
2     2  李四  67  2025-02-10   肺炎
3     3  王五  52  2025-01-20  糖尿病
4     4  赵六  38         NaN  NaN

=== 右连接 (Right) - 保留所有就诊记录 ===
   患者ID   姓名    年龄        就诊日期   诊断
0     1   张三  45.0  2025-01-01   肺炎
1     2   李四  67.0  2025-01-05  高血压
2     2   李四  67.0  2025-02-10   肺炎
3     3   王五  52.0  2025-01-20  糖尿病
4     5  NaN   NaN  2025-03-01   骨折

=== 外连接 (Outer) - 全部保留 ===
   患者ID   姓名    年龄        就诊日期   诊断
0     1   张三  45.

In [18]:
#apply 是 Pandas 里极其常用的方法，专门用来对每一行或每一列应用一个自定义函数。它的核心价值在于：当内置的 mean/sum 等方法解决不了你的需求时，你需要用 apply 走“自定义逻辑”。

# 案例一：在医疗项目里，你经常需要把连续数值（年龄）切成几个档位（青年/中年/老年）
#定义一个分类函数
def age_group(age):
    if age<40:
        return '青年'
    elif age<60:
        return '中年'
    else:
        return '老年'

#用apply应用到年龄列
patients['年龄分组']=patients['年龄'].apply(age_group)
print(patients)

print()
# 案例二：如果你将来的医疗数据里有“诊断”文本，你需要从中提取特征（如是否包含“肺炎”一词），用 apply 非常适合。
def has_pneumonia(diagnosis):
    return 1 if '肺炎' in diagnosis else 0

visits['是否肺炎']=visits['诊断'].apply(has_pneumonia)
visits

   患者ID  姓名  年龄 年龄分组
0     1  张三  45   中年
1     2  李四  67   老年
2     3  王五  52   中年
3     4  赵六  38   青年



,患者ID,就诊日期,诊断,是否肺炎
0,1,2025-01-01,肺炎,1
1,2,2025-01-05,高血压,0
2,2,2025-02-10,肺炎,1
3,3,2025-01-20,糖尿病,0
4,5,2025-03-01,骨折,0


In [24]:
#pivot_table 的核心价值在于：把“长格式”数据变成“宽格式”特征表，方便你把多维度的临床指标整合成一行一行的患者特征，直接送入机器学习模型。


# 场景1：患者多次随访的血压记录（长格式)
bp_data=pd.DataFrame({
    '患者ID':[1,1,2,2,3],
    '随访时间':['第1次','第2次','第3次','第4次','第5次'],
    '收缩压':[120, 125, 140, 150, 130],
    '舒张压':[80, 82, 90, 95, 85]
})

print("长格式数据（每个患者有多行）：")
print(bp_data)

# 使用 pivot_table 转为宽格式（每个患者一行）
bp_wide=bp_data.pivot_table(
    index='患者ID',
    columns='随访时间',
    values=['收缩压','舒张压'],
    aggfunc='mean'
)

print("\n宽格式数据（每个患者一行，特征分散到各列）：")
print(bp_wide)

#肠镜2：患者多次就诊的检验指标（长格式）
lab_data = pd.DataFrame({
    '患者ID': [1, 1, 2, 2, 3, 3],
    '检验项目': ['血糖', '血脂', '血糖', '血脂', '血糖', '血脂'],
    '结果': [5.6, 4.2, 7.8, 5.1, 6.0, 4.8]
})

print("=== 原始检验数据（长格式） ===")
print(lab_data)

# 用 pivot_table 转成宽格式：每个患者一行，血糖和血脂作为两列
lab_pivot = lab_data.pivot_table(
    index='患者ID',
    columns='检验项目',
    values='结果',
    aggfunc='mean'
)

print("\n=== 转换后（每个患者一行，特征为检验项目） ===")
print(lab_pivot)

长格式数据（每个患者有多行）：
   患者ID 随访时间  收缩压  舒张压
0     1  第1次  120   80
1     1  第2次  125   82
2     2  第3次  140   90
3     2  第4次  150   95
4     3  第5次  130   85

宽格式数据（每个患者一行，特征分散到各列）：
        收缩压                               舒张压                        
随访时间    第1次    第2次    第3次    第4次    第5次   第1次   第2次   第3次   第4次   第5次
患者ID                                                                 
1     120.0  125.0    NaN    NaN    NaN  80.0  82.0   NaN   NaN   NaN
2       NaN    NaN  140.0  150.0    NaN   NaN   NaN  90.0  95.0   NaN
3       NaN    NaN    NaN    NaN  130.0   NaN   NaN   NaN   NaN  85.0
=== 原始检验数据（长格式） ===
   患者ID 检验项目   结果
0     1   血糖  5.6
1     1   血脂  4.2
2     2   血糖  7.8
3     2   血脂  5.1
4     3   血糖  6.0
5     3   血脂  4.8

=== 转换后（每个患者一行，特征为检验项目） ===
检验项目   血糖   血脂
患者ID          
1     5.6  4.2
2     7.8  5.1
3     6.0  4.8
